# Notebook 2: The Fool's Gold Problem — ROI Analysis
Merges: scholarships (03), pro pathway (04), ROI summary (05), MLS salary (07).

Outputs: `scholarship_pool_by_div.png`, `avg_scholarship_by_div.png`, `player_funnel.png`,
`salary_by_league.png`, `mls_salary_distribution_2026.png`, `mls_club_payroll_2026.png`,
`mls_earnings_vs_investment.png`, `mls_top20_earners_2026.png`, `monte_carlo_tier3.png`,
`roi_by_tier.png`, `roi_analysis.csv`, `probability_matrix.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA = '../data/processed/'

costs  = pd.read_csv(DATA + 'costs.csv')
schols = pd.read_csv(DATA + 'scholarship_outcomes.csv')
pros   = pd.read_csv(DATA + 'pro_outcomes.csv')
mls_df = pd.read_csv(DATA + 'mls_player_salaries_2026.csv')

print(f'costs: {costs.shape}  schols: {schols.shape}  pros: {pros.shape}')
print(f'mls_df: {mls_df.shape} — {mls_df["club"].nunique()} clubs')

## 1. Scholarship Pool — How Many Spots Exist?

In [ ]:
s2024 = schols[schols['year'] == 2024].copy()

pivot = s2024.pivot_table(values='total_scholarship_pool_usd',
                          index='division', columns='gender', aggfunc='sum')
print('Total scholarship pool (USD) by division and gender (2024):')
print(pivot.map(lambda v: f'${v:,.0f}' if pd.notna(v) else '-'))

pivot.plot(kind='bar', color=['#e74c3c','#3498db'], alpha=0.85)
plt.title('Total Soccer Scholarship Pool by Division & Gender (2024)', fontsize=13)
plt.ylabel('USD')
plt.xticks(rotation=0)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v/1e6:.0f}M'))
plt.tight_layout()
plt.savefig('../data/analysis/scholarship_pool_by_div.png', dpi=150)
plt.show()

## 2. Scholarship Probability — What Are the Odds?

In [ ]:
total_youth_players = 4_000_000

s2024_soccer = s2024[s2024['sport'].isin(['M Soccer','W Soccer'])].copy()
total_roster_spots = s2024_soccer['roster_spots_total'].sum()
total_schol_spots  = (s2024_soccer['roster_spots_total'] *
                      (s2024_soccer['partial_scholarship_pct'] +
                       s2024_soccer['full_scholarship_pct']) / 100).sum().round(0)

prob_any_college = total_roster_spots / total_youth_players * 100
prob_any_schol   = total_schol_spots  / total_youth_players * 100

print(f'Total youth soccer players (est):    {total_youth_players:,}')
print(f'Total college soccer roster spots:   {total_roster_spots:,.0f}')
print(f'Total spots with any scholarship:    {total_schol_spots:,.0f}')
print(f'Prob any player plays college soccer: {prob_any_college:.2f}%')
print(f'Prob any player gets a scholarship:   {prob_any_schol:.2f}%')

In [ ]:
# Average scholarship value by division
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, gender in zip(axes, ['Men', 'Women']):
    sub = s2024[s2024['gender'] == gender].sort_values('avg_scholarship_value_usd', ascending=False)
    ax.barh(sub['division'], sub['avg_scholarship_value_usd'], color='#3498db', alpha=0.85)
    ax.set_title(f'Avg Scholarship Value — {gender} Soccer (2024)', fontsize=12)
    ax.set_xlabel('USD')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v:,.0f}'))

plt.tight_layout()
plt.savefig('../data/analysis/avg_scholarship_by_div.png', dpi=150)
plt.show()

In [ ]:
# Net cost after scholarship — does D1 scholarship cover Tier 3 investment?
tier3_career_cost = costs[costs['tier'] == 3]['total_mid'].mean() * 7

d1_men_schol   = schols[(schols['division']=='D1') & (schols['gender']=='Men')   & (schols['year']==2024)]['avg_scholarship_value_usd'].values[0] * 4
d1_women_schol = schols[(schols['division']=='D1') & (schols['gender']=='Women') & (schols['year']==2024)]['avg_scholarship_value_usd'].values[0] * 4
d2_men_schol   = schols[(schols['division']=='D2') & (schols['gender']=='Men')   & (schols['year']==2024)]['avg_scholarship_value_usd'].values[0] * 4

print('Scholarship return vs Tier 3 career investment:')
print(f'  Tier 3 career cost (7yr mid):  ${tier3_career_cost:,.0f}')
print(f'  D1 Men  scholarship (4yr avg): ${d1_men_schol:,.0f}  ->  net: ${d1_men_schol - tier3_career_cost:,.0f}')
print(f'  D1 Women scholarship (4yr avg):${d1_women_schol:,.0f}  ->  net: ${d1_women_schol - tier3_career_cost:,.0f}')
print(f'  D2 Men  scholarship (4yr avg): ${d2_men_schol:,.0f}  ->  net: ${d2_men_schol - tier3_career_cost:,.0f}')

## 3. Player Funnel — From 4M to 600 First-Division Spots

In [ ]:
funnel = pd.DataFrame([
    {'stage': 'All youth soccer players',   'count': 4_000_000},
    {'stage': 'Competitive travel (Tier 2)', 'count': 1_200_000},
    {'stage': 'ECNL / MLS NEXT (Tier 3)',   'count': 150_000},
    {'stage': 'Pro Academies (Tier 4)',      'count': 20_000},
    {'stage': 'Any pro contract (US)',       'count': 2_500},
    {'stage': 'MLS/NWSL first division',     'count': 600},
    {'stage': 'Top European league',         'count': 50},
])

funnel['prob_from_start'] = funnel['count'] / funnel['count'].iloc[0] * 100
print(funnel[['stage','count','prob_from_start']].to_string(index=False))
funnel.to_csv('../data/analysis/probability_matrix.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(funnel['stage'][::-1], funnel['count'][::-1], color='#2980b9', alpha=0.85)
ax.set_xscale('log')
ax.set_title('Youth Soccer to Professional: Player Funnel (Log Scale)', fontsize=12)
ax.set_xlabel('Number of Players (log scale)')
for i, (count, stage) in enumerate(zip(funnel['count'][::-1], funnel['stage'][::-1])):
    ax.text(count * 1.1, i, f'{count:,}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../data/analysis/player_funnel.png', dpi=150)
plt.show()

## 4. MLS Salary Distribution (2026 Real Data — MLSPA Spring Guide)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(mls_df['base_salary'], bins=60, color='#2980b9', alpha=0.8, edgecolor='white')
axes[0].set_xscale('log')
axes[0].axvline(mls_df['base_salary'].median(), color='red', linestyle='--',
                label=f'Median: ${mls_df["base_salary"].median():,.0f}')
axes[0].axvline(mls_df['base_salary'].mean(), color='orange', linestyle='--',
                label=f'Mean: ${mls_df["base_salary"].mean():,.0f}')
axes[0].set_title('MLS Base Salary Distribution (Log Scale)\n2026 Season — 916 Players', fontsize=12)
axes[0].set_xlabel('Base Salary (USD, log scale)')
axes[0].set_ylabel('Number of Players')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda v,_: f'${v/1e6:.1f}M' if v >= 1e6 else f'${v/1e3:.0f}k'))
axes[0].legend()

bins = [0, 88025, 113400, 250000, 500000, 1000000, 3000000, 30000000]
labels = ['Min\n$88k', 'Reserve\n$113k', '$114k–\n$250k', '$251k–\n$500k',
          '$501k–\n$1M', '$1M–\n$3M', '$3M+']
mls_df['salary_band'] = pd.cut(mls_df['base_salary'], bins=bins, labels=labels)
counts = mls_df['salary_band'].value_counts().sort_index()
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db','#9b59b6','#1abc9c']
bars = axes[1].bar(counts.index, counts.values, color=colors, alpha=0.85, edgecolor='white')
axes[1].set_title('Players by Salary Tier\n2026 MLS Season', fontsize=12)
axes[1].set_ylabel('Number of Players')
for bar, val in zip(bars, counts.values):
    pct = val / len(mls_df) * 100
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                f'{val}\n({pct:.0f}%)', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/analysis/mls_salary_distribution_2026.png', dpi=150)
plt.show()

print('Key MLS stats (2026):')
for label, val in [
    ('Minimum salary',      mls_df['base_salary'].min()),
    ('Median',              mls_df['base_salary'].median()),
    ('Mean',                mls_df['base_salary'].mean()),
    ('90th percentile',     mls_df['base_salary'].quantile(0.90)),
    ('Maximum (Messi)',     mls_df['base_salary'].max()),
    ('Total league payroll',mls_df['base_salary'].sum()),
]:
    print(f'  {label:<25}: ${val:>15,.0f}')

## 5. Club Payroll Comparison

In [ ]:
club = mls_df.groupby('club').agg(
    players=('base_salary','count'),
    total_payroll=('base_salary','sum'),
    median_salary=('base_salary','median'),
).sort_values('total_payroll', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

colors_c = ['#e74c3c' if c == 'Inter Miami' else '#3498db' for c in club.index]
axes[0].barh(club.index[::-1], club['total_payroll'][::-1] / 1e6, color=colors_c[::-1], alpha=0.85)
axes[0].set_xlabel('Total Payroll (USD millions)')
axes[0].set_title('MLS Club Total Payroll (2026)\nRed = Inter Miami (Messi effect)', fontsize=12)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:.0f}M'))

club_med = club.sort_values('median_salary', ascending=False)
axes[1].barh(club_med.index[::-1], club_med['median_salary'][::-1] / 1e3, color='#27ae60', alpha=0.85)
axes[1].set_xlabel('Median Player Salary (USD thousands)')
axes[1].set_title('MLS Club Median Player Salary (2026)', fontsize=12)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:.0f}k'))

plt.tight_layout()
plt.savefig('../data/analysis/mls_club_payroll_2026.png', dpi=150)
plt.show()

## 6. Reality Check — MLS Career Earnings vs Youth Investment

In [ ]:
tier3_career_cost = costs[costs['tier'] == 3]['total_mid'].mean() * 7
tier3_career_high = costs[costs['tier'] == 3]['total_high'].mean() * 7

# Watanabe (2010): avg 2.4yr initial career, ~6.6yr if reach yr4
scenarios = {
    'At MLS minimum\n(2.4yr career)':   88025 * 2.4,
    'At MLS median\n(2.4yr career)':    325000 * 2.4,
    'At MLS median\n(6.6yr career)':    325000 * 6.6,
    'At MLS mean\n(6.6yr career)':      mls_df['base_salary'].mean() * 6.6,
    'At $1M (top 13%)\n(6.6yr career)': 1_000_000 * 6.6,
}

fig, ax = plt.subplots(figsize=(10, 5))
sce_values = list(scenarios.values())
bar_colors = ['#e74c3c' if v < tier3_career_cost else '#27ae60' for v in sce_values]

ax.barh(list(scenarios.keys()), [v/1e3 for v in sce_values], color=bar_colors, alpha=0.85)
ax.axvline(tier3_career_cost/1e3, color='black', linestyle='--', linewidth=2,
           label=f'Tier 3 investment (mid): ${tier3_career_cost:,.0f}')
ax.axvline(tier3_career_high/1e3, color='gray', linestyle=':', linewidth=1.5,
           label=f'Tier 3 investment (high): ${tier3_career_high:,.0f}')
ax.set_xlabel('Total Career Earnings (USD thousands)')
ax.set_title('MLS Career Earnings vs Youth Soccer Investment\n(Red = does NOT recoup Tier 3 cost)', fontsize=12)
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:.0f}k'))
for i, val in enumerate(sce_values):
    ax.text(val/1e3 + 5, i, f'${val/1e3:,.0f}k', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../data/analysis/mls_earnings_vs_investment.png', dpi=150)
plt.show()

break_even_salary = tier3_career_cost / 2.4
pct_above = (mls_df['base_salary'] >= break_even_salary).mean() * 100
print(f'Tier 3 investment (7yr mid): ${tier3_career_cost:,.0f}')
print(f'Break-even salary (2.4yr):   ${break_even_salary:,.0f}/yr')
print(f'% of MLS players at or above: {pct_above:.0f}%')

## 7. Salary by League — The Wider Context

In [ ]:
leagues_of_interest = ['MLS', 'NWSL', 'USL Championship', 'Bundesliga', 'Premier League', 'La Liga']
league_data = pros[pros['league'].isin(leagues_of_interest)].copy()

fig, ax = plt.subplots(figsize=(12, 5))
league_order = league_data.groupby('league')['salary_usd_mid'].mean().sort_values(ascending=False).index

for i, league in enumerate(league_order):
    row = league_data[league_data['league'] == league]
    lo  = row['salary_usd_low'].mean()
    mid = row['salary_usd_mid'].mean()
    hi  = row['salary_usd_high'].mean()
    ax.bar(i, mid, color='#3498db', alpha=0.8)
    ax.errorbar(i, mid, yerr=[[mid-lo],[hi-mid]], fmt='none', color='black', capsize=5)

ax.set_xticks(range(len(league_order)))
ax.set_xticklabels(league_order, rotation=15, ha='right')
ax.set_title('Median Salary by League — US Player Context (2024)', fontsize=13)
ax.set_ylabel('USD (mid estimate)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v:,.0f}'))
plt.tight_layout()
plt.savefig('../data/analysis/salary_by_league.png', dpi=150)
plt.show()

## 8. Expected Value by Tier — The Core ROI Analysis

In [ ]:
career_cost = {
    1: costs[costs['tier']==1]['total_mid'].mean() * 13,
    2: costs[costs['tier']==2]['total_mid'].mean() * 9,
    3: costs[costs['tier']==3]['total_mid'].mean() * 7,
    4: 0,  # Pro academies are free
}

d1_men_4yr   = schols[(schols['division']=='D1') & (schols['gender']=='Men')   & (schols['year']==2024)]['avg_scholarship_value_usd'].values[0] * 4
d1_women_4yr = schols[(schols['division']=='D1') & (schols['gender']=='Women') & (schols['year']==2024)]['avg_scholarship_value_usd'].values[0] * 4
d2_men_4yr   = schols[(schols['division']=='D2') & (schols['gender']=='Men')   & (schols['year']==2024)]['avg_scholarship_value_usd'].values[0] * 4
mls_5yr      = pros[(pros['league']=='MLS') & (pros['year']==2026)]['salary_usd_mid'].values[0] * 5
nwsl_5yr     = pros[(pros['league']=='NWSL')]['salary_usd_mid'].values[0] * 5

print('Career investment (mid est):')
for t, v in career_cost.items(): print(f'  Tier {t}: ${v:,.0f}')
print(f'\nReturn benchmarks: D1 Men 4yr=${d1_men_4yr:,.0f}  D1 Women 4yr=${d1_women_4yr:,.0f}')
print(f'                   MLS 5yr=${mls_5yr:,.0f}  NWSL 5yr=${nwsl_5yr:,.0f}')

In [ ]:
prob = {
    'D1 schol from T3':    0.08,
    'Any schol from T3':   0.35,
    'Pro from T3':         0.02,
    'Pro from T4':         0.15,
}

scenarios = []
for tier in [1, 2, 3, 4]:
    invest = career_cost[tier]
    for scenario, ret, p in [
        ('No outcome',         0,              1.00),
        ('Partial schol',      d2_men_4yr/2,   prob['Any schol from T3'] if tier==3 else 0.05),
        ('Full D1 schol',      d1_men_4yr,     prob['D1 schol from T3']  if tier==3 else 0.005),
        ('MLS career',         mls_5yr,        prob['Pro from T3'] if tier < 4 else prob['Pro from T4']),
    ]:
        net = ret - invest
        scenarios.append({'tier': tier, 'scenario': scenario, 'investment': invest,
                          'return': ret, 'net': net, 'probability': p, 'ev_contribution': p * net})

df_scen = pd.DataFrame(scenarios)
ev_by_tier = df_scen.groupby('tier')['ev_contribution'].sum().reset_index()
ev_by_tier.columns = ['tier','expected_value_net_usd']

print('Expected net value (return minus cost) by tier:')
for _, row in ev_by_tier.iterrows():
    print(f'  Tier {int(row["tier"])}: ${row["expected_value_net_usd"]:,.0f}')

df_scen.to_csv('../data/analysis/roi_analysis.csv', index=False)

## 9. Monte Carlo Simulation — 100,000 Tier 3 Players

In [ ]:
np.random.seed(42)
N = 100_000

cost_low  = costs[costs['tier']==3]['total_low'].mean()  * 7
cost_high = costs[costs['tier']==3]['total_high'].mean() * 7
sim_costs = np.random.uniform(cost_low, cost_high, N)

rand = np.random.random(N)
sim_returns = np.where(rand < 0.005, mls_5yr,
              np.where(rand < 0.08,  d1_men_4yr,
              np.where(rand < 0.35,  d2_men_4yr/2,
                                     0)))
sim_net = sim_returns - sim_costs

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(sim_net, bins=80, color='#2980b9', alpha=0.75, edgecolor='white')
ax.axvline(sim_net.mean(), color='red', linestyle='--', label=f'Mean: ${sim_net.mean():,.0f}')
ax.axvline(0, color='black', linestyle='-', linewidth=1.5, label='Break-even')
ax.set_title('Monte Carlo: Net Financial Outcome for Tier 3 Player (100k simulations)', fontsize=12)
ax.set_xlabel('Net Return (USD)')
ax.set_ylabel('Frequency')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v:,.0f}'))
ax.legend()
plt.tight_layout()
plt.savefig('../data/analysis/monte_carlo_tier3.png', dpi=150)
plt.show()

print(f'Monte Carlo summary (Tier 3):')
print(f'  Mean net outcome:   ${sim_net.mean():,.0f}')
print(f'  Median net outcome: ${np.median(sim_net):,.0f}')
print(f'  % who break even:   {(sim_net >= 0).mean()*100:.1f}%')
print(f'  % who go pro:       {(sim_returns >= mls_5yr).mean()*100:.2f}%')

## 10. ROI Waterfall — Expected Net Value by Tier

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
tier_labels = ['Tier 1\nRec', 'Tier 2\nTravel', 'Tier 3\nECNL', 'Tier 4\nPro Academy']
ev_vals = ev_by_tier['expected_value_net_usd'].values
colors = ['#e74c3c' if v < 0 else '#27ae60' for v in ev_vals]

ax.bar(tier_labels, ev_vals, color=colors, alpha=0.85, edgecolor='white')
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Expected Net Financial Value of Youth Soccer Investment by Tier', fontsize=12)
ax.set_ylabel('Expected Net USD')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v:,.0f}'))
for i, v in enumerate(ev_vals):
    ax.text(i, v + (200 if v >= 0 else -1500), f'${v:,.0f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/analysis/roi_by_tier.png', dpi=150)
plt.show()